# CNN Example

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import random

# Reproducibility
random.seed(42)
torch.manual_seed(42)

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print("Torch:", torch.__version__, "| Device:", device)


## Data

In [14]:
# Create random images and labels to simulate training and test data. Dataset size=128
train_images = torch.randn(128, 3, 32, 32)
train_labels = torch.randint(0, 10, (128,))

test_images = torch.randn(128, 3, 32, 32)
test_labels = torch.randint(0, 10, (128,))

# Data normalization
train_images = (train_images - 0.5) / 0.5
test_images = (test_images - 0.5) / 0.5

# Create datasets
train_dataset = TensorDataset(train_images, train_labels)
test_dataset = TensorDataset(test_images, test_labels)

# Create dataloaders with batch size 32
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Training set size: {len(train_dataset)} Samples")
print(f"Test set size: {len(test_dataset)} Samples")

Training set size: 128 Samples
Test set size: 128 Samples


## Model

In [18]:
def block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
    )

class MyCNN(nn.Module):
    def __init__(self):
        super(MyCNN, self).__init__()
        # three base blocks with projection shortcuts when channels change
        self.b1 = block(3, 32)
        self.s1 = nn.Conv2d(3, 32, 1, bias=False)
        self.p1 = nn.MaxPool2d(2)  # 32x32 -> 16x16
        self.b2 = block(32, 64)
        self.s2 = nn.Conv2d(32, 64, 1, bias=False)
        self.p2 = nn.MaxPool2d(2)  # 16x16 -> 8x8
        self.b3 = block(64, 128)
        self.s3 = nn.Conv2d(64, 128, 1, bias=False)
        self.p3 = nn.MaxPool2d(2)  # 8x8 -> 4x4
        self.relu = nn.ReLU(inplace=True) # https://discuss.pytorch.org/t/guidelines-for-when-and-why-one-should-set-inplace-true/50923
        self.fc = nn.Linear(128*4*4, 10)  # logits for 10 classes

    def forward(self, x: torch.tensor):
        y = self.b1(x) + self.s1(x)
        y = self.relu(y)
        y = self.p1(y)
        y = self.b2(y) + self.s2(y)
        y = self.relu(y)
        y = self.p2(y)
        y = self.b3(y) + self.s3(y)
        y = self.relu(y)
        y = self.p3(y)
        y = y.view(y.size(0), -1)
        logits = self.fc(y)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MyCNN().to(device)
print(f"Number of parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")


Number of parameters: 318282


## Accuracy helpers

In [23]:
@torch.no_grad() # function with this decorator will not create any computational graph; save memory
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, pred = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (pred == labels).sum().item()
    return 100 * correct / total


In [25]:
num_epochs = 10
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

train_acc_list = []
test_acc_list = []

for epoch in range(num_epochs):
    model.train()
    correct = 0
    total = 0
    for iter_no, (images, labels) in enumerate(train_loader, start=1):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        pred = logits.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total += labels.numel()
        print(f"Epoch [{epoch+1}/{num_epochs}], Iter [{iter_no+1}], Train Acc: {correct/total*100:.2f}%")

    test_acc = evaluate(model, test_loader)
    test_acc_list.append(test_acc)
    print(f"Epoch [{epoch+1}/{num_epochs}], Test Acc: {test_acc:.2f}%")



Epoch [1/10], Iter [2], Train Acc: 100.00%
Epoch [1/10], Iter [3], Train Acc: 100.00%
Epoch [1/10], Iter [4], Train Acc: 100.00%
Epoch [1/10], Iter [5], Train Acc: 100.00%
Epoch [1/10], Test Acc: 7.03%
Epoch [2/10], Iter [2], Train Acc: 100.00%
Epoch [2/10], Iter [3], Train Acc: 100.00%
Epoch [2/10], Iter [4], Train Acc: 100.00%
Epoch [2/10], Iter [5], Train Acc: 100.00%
Epoch [2/10], Test Acc: 10.16%
Epoch [3/10], Iter [2], Train Acc: 100.00%
Epoch [3/10], Iter [3], Train Acc: 100.00%
Epoch [3/10], Iter [4], Train Acc: 100.00%
Epoch [3/10], Iter [5], Train Acc: 100.00%
Epoch [3/10], Test Acc: 7.81%
Epoch [4/10], Iter [2], Train Acc: 100.00%
Epoch [4/10], Iter [3], Train Acc: 100.00%
Epoch [4/10], Iter [4], Train Acc: 100.00%
Epoch [4/10], Iter [5], Train Acc: 100.00%
Epoch [4/10], Test Acc: 7.81%
Epoch [5/10], Iter [2], Train Acc: 100.00%
Epoch [5/10], Iter [3], Train Acc: 100.00%
Epoch [5/10], Iter [4], Train Acc: 100.00%
Epoch [5/10], Iter [5], Train Acc: 100.00%
Epoch [5/10], Test 

## Plot training curve

In [ ]:
plt.figure()
plt.plot(range(1, num_epochs+1), train_acc_list, label='Train Accuracy')
plt.plot(range(1, num_epochs+1), test_acc_list, label='Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)
plt.title('Training Curve')
plt.show()
